# Stage 18B: independent branch-group confirmation

Stage 18Aで良好だったbranch-group除外 + visible-prefix calibration + GR + 20% blendを、Stage 18Aとは重ならない別の固定150 cutsで再検証します。係数探索は行いません。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import json, os, shutil, subprocess
REPOSITORY_URL = 'https://github.com/Okada-N13/rogii.git'
repo_dir = Path('/content/ROGII')
drive_root = Path('/content/drive/MyDrive/kaggle/rogii')
artifact_dir = drive_root / 'artifacts'
data_dir = drive_root / 'data'
if not (repo_dir / '.git').is_dir():
    subprocess.run(['git', 'clone', REPOSITORY_URL, str(repo_dir)], check=True)
else:
    subprocess.run(['git', '-C', str(repo_dir), 'pull', '--ff-only', 'origin', 'main'], check=True)
if shutil.which('uv') is None:
    subprocess.run(['bash', '-lc', 'curl -LsSf https://astral.sh/uv/install.sh | sh'], check=True)
os.environ['PATH'] = '/root/.local/bin:' + os.environ['PATH']
subprocess.run(['uv', 'sync', '--frozen'], cwd=repo_dir, check=True)
assert (data_dir / 'train').is_dir()
artifact_dir.mkdir(parents=True, exist_ok=True)
def run_checked(command):
    result = subprocess.run(command, cwd=repo_dir, text=True, capture_output=True)
    if result.stdout:
        print(result.stdout, flush=True)
    if result.returncode != 0:
        print(result.stderr, flush=True)
        raise RuntimeError(f'Command failed with exit code {result.returncode}: {command}')
    return result
subprocess.run(['git', '-C', str(repo_dir), 'rev-parse', '--short', 'HEAD'], check=True)

## 固定artifact

Stage 16B v003、Stage 17A v002、Stage 17B v001を再利用します。CPUで実行できます。

In [ ]:
stage16b_run = artifact_dir / 'stage16b_testlike_validation_full_v003'
stage17a_run = artifact_dir / 'stage17_public_replay_full_v002'
stage17b_run = artifact_dir / 'stage17b_selector_replay_full_v001'
assert (stage16b_run / 'donor_graph.parquet').is_file()
assert (stage17a_run / 'replay_predictions.parquet').is_file()
assert (stage17b_run / 'selector_predictions.parquet').is_file()
print(stage16b_run, stage17a_run, stage17b_run, sep='\n')

## 独立150-cut確認

各fold × prefix fractionのhash順位6–11を使用します。Stage 18Aの順位0–5とは完全に非重複です。primaryはbranch-group除外、profileは`prefix_gr_w020`のみです。

In [ ]:
RUN_ID = 'stage18b_branch_confirmation_full_v001'
run_dir = artifact_dir / RUN_ID
if not (run_dir / 'summary.json').is_file():
    run_checked([
        'uv', 'run', 'rogii-branch-retrieval',
        '--config', 'configs/experiment/stage18b_branch_confirmation.yaml',
        '--stage16b-run', str(stage16b_run), '--stage17a-run', str(stage17a_run),
        '--stage17b-run', str(stage17b_run), '--data-dir', str(data_dir),
        '--artifact-dir', str(artifact_dir), '--run-id', RUN_ID,
    ])
else:
    print('Reusing completed run:', run_dir)
summary = json.loads((run_dir / 'summary.json').read_text())
{
    'stage18b_complete': summary['stage18b_complete'],
    'promoted_to_all_cut_retrieval': summary['promoted_to_all_cut_retrieval'],
    'sample_cuts': summary['sample_cuts'],
    'sample_offset_per_stratum': summary['sample_offset_per_stratum'],
    'sample_sha256': summary['sample_sha256'],
    'reference_overlap_cuts': summary['reference_overlap_cuts'],
    'primary_family': summary['primary_family'],
    'primary_profile': summary['primary_profile'],
    'primary_report': summary['reports']['branch__prefix_gr_w020'],
    'bootstrap_95pct': summary['bootstrap_95pct'],
    'gates': summary['gates'],
    'next_step': summary['next_step'],
}

最後の辞書を共有してください。`promoted_to_all_cut_retrieval`がtrueのときだけ、全cut化へ進みます。